## Replication of Hmielowski et al.: Media Trust in Rural vs. Urban Communities

**Paper:** Hmielowski, J. D., Heffron, E., Ma, Y., & Munroe, M. A. (2025). *You’ve lost that trusting feeling: Diminishing trust in the news media in rural versus urban US communities.* The Social Science Journal, 62 (3), 850–864. Published online in 2021. DOI: 10.1080/03623319.2021.1969510.

The study uses the General Social Survey (GSS) to examine the relationship between political ideology, residential location, time, and trust in the news media.

**Goal:** Re-estimate the paper's four ordinal logistic regression models (Table 1) and reproduce its main three predicted-probability figures.

- **Dependent variable:**
  - `CONPRESS` Confidence in the press (0 = hardly any trust, 1 = only some, 2 = a great deal).
- **Independent variables:**
  - `POLVIEWS` — Political ideology (0 = extremely liberal to 6 = extremely conservative).
  - `SRCBELT` — Residential location (0 = rural to 5 = largest metropolitan areas).
  - `YEAR` — Survey year, recoded as `YEAR - 1972`.
- **Control variables:**
  - `AGE`, `EDUC`, `INCOME`, `SEX`, and race/ethnicity (`RACE` and `HISPANIC`).

The paper examines the main effects and interactions among ideology, residential location, and time using ordinal logistic regression.

In [1]:
# Loading packages
library(haven)
library(MASS)     
library(ggplot2) 

### 1. Read the GSS data

The analysis covers the period from 1975 to 2018.

In [2]:
dir.create("data", showWarnings = FALSE)
unzip("GSS_stata.zip", files = "GSS_stata/gss7224_r3a.dta", exdir = "data")
variables <- c("year", 
               "conpress", 
               "polviews", 
               "srcbelt", 
               "income", 
               "educ", 
               "race", 
               "hispanic",
               "sex",
               "age",
               "wtssall")
gss <- read_dta("data/GSS_stata/gss7224_r3a.dta",
                col_select = tidyselect::any_of(variables))
gss[] <- lapply(gss, as.numeric)
write.csv(gss,"data/gss_selected_variables.csv",row.names = FALSE,na = "")
cat("Total observations:", nrow(gss), "\n")

Total observations: 75699 


In [3]:
dir.create("output", showWarnings = FALSE)
gss <- read.csv("data/gss_selected_variables.csv")
gss <- subset(gss, year >= 1975 & year <= 2018)
cat("GSS records in 1975-2018:", nrow(gss), "\n")

GSS records in 1975-2018: 60213 


### 2. Variables and data cleaning
The authors' variable definitions (paper pp. 855–856):

| Analysis variable | GSS item | Coding used in this notebook |
|---|---|---|
| Confidence in the press | `conpress` | `trust = 3 − conpress`: 0 hardly any; 1 only some; 2 a great deal |
| Ideology | `polviews` | `ideology = polviews − 1`: 0 extremely liberal to 6 extremely conservative |
| Residential location | `srcbelt` | `location = 6 − srcbelt`: 0 most rural to 5 largest metropolitan central city |
| Time | `year` | `time = year − 1972` |
| Controls | `sex`, `race`, `age`, `income`, `educ` | Female indicator; non-Hispanic white indicator; age; 12-category income; years of education |

Invalid/out-of-range responses are treated as missing.

In [4]:
valid <- function(x, low, high) ifelse(!is.na(x) & x >= low & x <= high, x, NA_real_)

gss$trust     <- 3 - valid(gss$conpress, 1, 3)
gss$ideology  <- valid(gss$polviews, 1, 7) - 1
gss$location  <- 6 - valid(gss$srcbelt, 1, 6)
gss$time      <- gss$year - 1972
gss$female    <- ifelse(gss$sex %in% 1:2, as.numeric(gss$sex == 2), NA_real_)
gss$white     <- ifelse(gss$race %in% 1:3, as.numeric(gss$race == 1), NA_real_)
gss$age       <- valid(gss$age, 18, 120)
gss$income    <- valid(gss$income, 1, 12)
gss$educ      <- valid(gss$educ, 0, 20)

model_vars <- c("trust", "ideology", "location", "time", 
                "female", "white", "age", "income", "educ")
dat <- gss[complete.cases(gss[, model_vars]), model_vars]
dat$trust <- ordered(dat$trust, levels = 0:2)

cat("Total observations for replication:", nrow(dat))

Total observations for replication: 34368

#### Sample Size Discrepancy

The final sample usd for analysis in the paper was reduced to 18,054 due to missing data and questions not being asked in certain survey years. My replication uses the 2024 GSS release resulting in a difference in observations. However, I retain all observations satisfying the specified inclusion criteria from the paper.

## 3. Descriptive statistics

In [5]:
summary(dat[, model_vars])
cat("\nTrust distribution (%)\n")
print(round(100 * prop.table(table(dat$trust)), 1))

 trust        ideology       location          time           female      
 0:11556   Min.   :0.00   Min.   :0.000   Min.   : 3.00   Min.   :0.0000  
 1:17704   1st Qu.:2.00   1st Qu.:1.000   1st Qu.:11.00   1st Qu.:0.0000  
 2: 5108   Median :3.00   Median :1.000   Median :22.00   Median :1.0000  
           Mean   :3.09   Mean   :2.015   Mean   :22.75   Mean   :0.5432  
           3rd Qu.:4.00   3rd Qu.:3.000   3rd Qu.:34.00   3rd Qu.:1.0000  
           Max.   :6.00   Max.   :5.000   Max.   :46.00   Max.   :1.0000  
     white             age            income           educ      
 Min.   :0.0000   Min.   :18.00   Min.   : 1.00   Min.   : 0.00  
 1st Qu.:1.0000   1st Qu.:31.00   1st Qu.: 9.00   1st Qu.:12.00  
 Median :1.0000   Median :43.00   Median :12.00   Median :12.00  
 Mean   :0.8136   Mean   :45.29   Mean   :10.12   Mean   :13.05  
 3rd Qu.:1.0000   3rd Qu.:58.00   3rd Qu.:12.00   3rd Qu.:15.00  
 Max.   :1.0000   Max.   :89.00   Max.   :12.00   Max.   :20.00  


Trust distribution (%)

   0    1    2 
33.6 51.5 14.9 


## 4. Ordinal logistic regressions — Table 1
The paper uses ordinal logistic regression, with higher outcome values meaning **more confidence**. All models include controls for sex, race, age, income and education.

- **Model 1:** Ideology + location + time (main effects).
- **Model 2:** Add ideology × time to Model 1.
- **Model 3:** Add location × time to Model 1.
- **Model 4:** Add ideology × time, location × time, and ideology × location × time to Model 1.

The `polr` coefficients are **log-odds**, as in the paper's Table 1.

In [6]:
controls <- "female + white + age + income + educ"
f1 <- as.formula(paste("trust ~ ideology + location + time +", controls))
f2 <- as.formula(paste("trust ~ ideology * time + location +", controls))
f3 <- as.formula(paste("trust ~ location * time + ideology +", controls))
f4 <- as.formula(paste("trust ~ ideology * location * time +", controls))

models <- list(
  Model_1 = polr(f1, data = dat, method = "logistic", Hess = TRUE),
  Model_2 = polr(f2, data = dat, method = "logistic", Hess = TRUE),
  Model_3 = polr(f3, data = dat, method = "logistic", Hess = TRUE),
  Model_4 = polr(f4, data = dat, method = "logistic", Hess = TRUE)
)

make_table <- function(fit, model_name) {
  a <- coef(summary(fit))
  a <- a[names(coef(fit)), , drop = FALSE]  # Regression slopes, not cutpoints
  data.frame(model = model_name, term = rownames(a),
             coefficient = a[, "Value"], se = a[, "Std. Error"],
             wald_chisq = a[, "t value"]^2,
             p_value = 2 * pnorm(-abs(a[, "t value"])), row.names = NULL)
}
results <- do.call(rbind, Map(make_table, models, names(models)))
print(results, row.names = FALSE, digits = 4)
write.csv(results, "output/logit_table1.csv", row.names = FALSE)

   model                   term coefficient        se wald_chisq    p_value
 Model_1               ideology  -0.1909551 0.0078678  5.890e+02 4.035e-130
 Model_1               location   0.0225066 0.0071541  9.897e+00  1.655e-03
 Model_1                   time  -0.0351357 0.0008812  1.590e+03  0.000e+00
 Model_1                 female  -0.0222529 0.0211537  1.107e+00  2.928e-01
 Model_1                  white  -0.1235252 0.0287217  1.850e+01  1.702e-05
 Model_1                    age   0.0009496 0.0006381  2.214e+00  1.367e-01
 Model_1                 income  -0.0315130 0.0043037  5.362e+01  2.436e-13
 Model_1                   educ  -0.0092765 0.0038166  5.908e+00  1.507e-02
 Model_2               ideology  -0.1128159 0.0158996  5.035e+01  1.289e-12
 Model_2                   time  -0.0250882 0.0019826  1.601e+02  1.062e-36
 Model_2               location   0.0228347 0.0071605  1.017e+01  1.428e-03
 Model_2                 female  -0.0239798 0.0211765  1.282e+00  2.575e-01
 Model_2    

## 5. Compare results - Table 1

In [7]:
published <- data.frame(model = c("Model_1", "Model_1", "Model_1",
                                  "Model_2", "Model_3", "Model_4"),
                        term = c("ideology", "location", "time",
                                 "ideology:time", "location:time",
                                 "ideology:location:time"),
                        paper_coef = c(-0.13, 0.02, -0.04,
                                       -0.006, 0.002, -0.001),
                        paper_wald = c(158.41, 2.76, 884.25,
                                       57.75, 9.39, 4.07))

comparison <- merge(published, results,by = c("model", "term"),all.x = TRUE,sort = FALSE)
comparison <- comparison[, c("model", "term",
                             "paper_coef", "coefficient",
                             "paper_wald", "wald_chisq", "p_value")]

names(comparison) <- c("Model", "Variable",
                       "Paper LogOR", "Replication LogOR",
                       "Paper Wald", "Replication Wald",
                       "Replication p-value")

comparison[, 3:4] <- round(comparison[, 3:4], 4)
comparison[, 5:6] <- round(comparison[, 5:6], 2)
comparison[, 7] <- ifelse(comparison[, 7] < 0.001, "<0.001", sprintf("%.3f", comparison[, 7]))
options(width = 200)
print(comparison, row.names = FALSE)
dir.create("output", showWarnings = FALSE)
write.csv(comparison,"output/published_vs_replication.csv",row.names = FALSE)

   Model               Variable Paper LogOR Replication LogOR Paper Wald Replication Wald Replication p-value
 Model_1               ideology      -0.130           -0.1910     158.41           589.05              <0.001
 Model_1               location       0.020            0.0225       2.76             9.90               0.002
 Model_1                   time      -0.040           -0.0351     884.25          1589.73              <0.001
 Model_2          ideology:time      -0.006           -0.0033      57.75            31.83              <0.001
 Model_3          location:time       0.002            0.0022       9.39            18.04              <0.001
 Model_4 ideology:location:time      -0.001            0.0007       4.07             3.52               0.061


## 6. Reproduce figures

In [8]:
# Hold controls at sample means
controls <- data.frame(female = mean(dat$female),
                       white = mean(dat$white),
                       age = mean(dat$age),
                       income = mean(dat$income),
                       educ = mean(dat$educ))

In [9]:
make_plot_data <- function(model, newdata, group_var) {
    probs <- predict(model, newdata = newdata, type = "probs")
    plot_data <- data.frame(year = rep(newdata$time + 1972, times = 3),
                            group = rep(newdata[[group_var]], times = 3),
                            response = rep(c("Hardly any", "Only some", "A great deal"), each = nrow(newdata)), probability = as.vector(probs))
    return(plot_data)
}

## 7. Results and conclusion 